# 01 — Data Audit

Purpose: before writing any loader, look at what's *actually* in the raw
CSE export -- how many files, how big, and what quirks each one has --
rather than assuming a clean, uniform layout across 34 files spanning
1985-2025.

This notebook is exploratory and read-only: it doesn't write to
`data/interim/` or `data/processed/`. The findings here are what
motivated the loader designs in `src/ingestion/` (documented again,
more concisely, in the project README's "Data quality issues found and
handled" section).

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append("..")

RAW_DIR = Path("../data/raw")
files = sorted(p for p in RAW_DIR.iterdir() if p.is_file())
print(f"{len(files)} files in the raw export\n")
for p in files:
    size_kb = p.stat().st_size / 1024
    print(f"{size_kb:9,.0f} KB  {p.name}")

6 files in the raw export

   15,276 KB  Daily Shares Price List -2021-2025.zip
    1,458 KB  Dividends.xls
       84 KB  Foreign Activity - Monthly.xlsx
      980 KB  Market Capitalisation of Listed Companies.xls
    2,668 KB  Market Indices - Daily.xls
      352 KB  Sector Market Capitalisation.xls


## Header row position varies by file

A generic "row 0 is the header" assumption breaks immediately. Compare
two files below: one has data starting almost immediately, another
needs several title/blank rows skipped first.

In [2]:
def peek(path, sheet_name=0, n=6):
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None, nrows=n)
    return raw

peek(RAW_DIR / "Sector Market Capitalisation.xls", sheet_name="2025")

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,NaN,Sector Market Capitalisation - Monthly 2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,SECTOR,January,February,March,April,May,June,July,August,September,October,November,December,October,November,December
3,NaN,Automobiles & Components,6906360000,6448080000,6504360000,6464160000,6874200000,7187760000,8301300000,7879200000,7629960000,NaN,NaN,NaN,7396800000,7332480000,7203840000
4,NaN,Banks,855819422651.75,828553985331.5,802069678094.949951,756466860194.199951,808373603104.099976,872347868498.199951,962425874235.949951,1059315910382.849976,1070504637840.25,NaN,NaN,NaN,1099549205826.149902,1110509398259.350098,1088258687988.800049
5,NaN,Capital Goods,1053014930287.199951,988183476828.900024,953578515443.050049,980279182485.150024,1033403883888.050049,1112587136976.5,1208370992599.899902,1206275484624.75,1237538198127.149902,NaN,NaN,NaN,1258501266156,1309100073365.25,1359157049873.449951


Row 0 here is a title row ("Sector Market Capitalisation - Monthly
2025"), not the header. Naively searching for a cell *containing* the
substring "sector" would incorrectly match this title row too (it was
mistaken for the real header by an early version of the parser) --
`src/ingestion/load_sector_data.py` instead requires an **exact**,
stripped match on `"SECTOR"`.

In [3]:
peek(RAW_DIR / "Market Indices - Daily.xls", sheet_name=0, n=6)

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Market Indices - Daily,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,S&P Sri Lanka 20,Banks Finance & Insurance,"Beverage, Food & Tobacco",Chemicals & Pharmaceuticals,Construction & Engineering,Diversified,Footwear & Textile,...,Manufacturing,Motors,Oil Palms,Plantations,Power & Energy,Services,NaN,Stores & Supplies,Telecommunications,Trading
4,NaN,All Share Price Index,Milanka Price Index,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1985-01-02 00:00:00,96.09,81.69,NaN,84.88,88.25,84.79,124.83,NaN,104.95,...,102.14,78.51,97.27,NaN,NaN,93.09,NaN,93.67,NaN,93.44


This file has a different quirk: the header is split across **two**
rows (row 3 has the S&P SL20 + sector index names, row 4 has ASPI /
Milanka labels only in columns 1-2). A single-row header detector picks
whichever row has more non-null cells -- which is row 3, silently
losing the ASPI and Milanka column names entirely. `load_indices.py`
merges both header rows explicitly instead.

## Known placeholder / bad-data values

A few cells contain non-numeric strings in place of a number, standing
in for "no trading happened" rather than a missing value. If these
aren't handled, `pd.to_numeric` would raise, or a naive `.astype(float)`
would fail outright.

In [4]:
fa_raw = pd.read_excel(RAW_DIR / "Foreign Activity - Monthly.xlsx", sheet_name=0, header=None)
row = fa_raw.iloc[3, 1:]
non_numeric = row[pd.to_numeric(row, errors="coerce").isna() & row.notna()]
print("Non-numeric cells in the 'Purchases / Foreign Companies' row:")
print(non_numeric)

Non-numeric cells in the 'Purchases / Foreign Companies' row:
Series([], Name: 3, dtype: object)


That's the March 2020 COVID market-closure cell. `pd.to_numeric(...,
errors="coerce")` turns this (and any other stray text) into `NaN`,
which then gets dropped like any other missing value -- see
`src/cleaning/tidy.py:coerce_numeric` and the equivalent inline handling
in `load_foreign_activity.py`.

## Column position drift across decades

The `SECTOR` column doesn't sit at a fixed index across all year sheets
in the same workbook -- some layouts have a leading blank spacer column,
others don't.

In [5]:
for sheet in ["2025", "2006"]:
    raw = pd.read_excel(RAW_DIR / "Sector Market Capitalisation.xls", sheet_name=sheet, header=None)
    header_row = None
    for i in range(min(8, len(raw))):
        if raw.iloc[i].astype(str).str.strip().str.upper().eq("SECTOR").any():
            header_row = i
            break
    header = raw.iloc[header_row]
    sector_pos = next(i for i, c in enumerate(header) if pd.notna(c) and str(c).strip().upper() == "SECTOR")
    print(f"sheet {sheet}: header row = {header_row}, SECTOR column at position {sector_pos}")

sheet 2025: header row = 2, SECTOR column at position 1
sheet 2006: header row = 2, SECTOR column at position 0


Confirmed: the position shifts between the two eras. `load_sector_data.py`
locates the `SECTOR` column dynamically per sheet rather than assuming a
fixed index -- see `_load_year_sheet()`.

## Takeaways that shaped the loader design

1. Build one reusable header-detection utility (`excel_parser.py`)
   rather than hand-coding every file, but let files with a genuinely
   different structure (two-row header, fixed section map) opt out of
   it and handle themselves explicitly.
2. Always coerce to numeric with `errors="coerce"` rather than assuming
   clean numeric cells.
3. Never assume a fixed column index for a named field -- locate it by
   matching the header value.
4. Match header labels **exactly**, not by substring, since title rows
   can accidentally contain the same words as real header rows.

See the README's "Data quality issues found and handled" section for
the four more issues (duplicate rows, share-class duplicates, sector
name drift, an implausible event-study outlier) that surfaced later
downstream, in `src/features/`, rather than at the raw-parsing stage
audited here.